In [1]:
#detecting noisy data
#utility function
#Use for preprocessing kaggle: proposed3_preprocessing
import json
import pandas as pd
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

def save_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

In [3]:
import re
from collections import Counter
from langdetect import detect, LangDetectException

In [ ]:

def punctuation_noise(text):
    punct = len(re.findall(r"[^\w\s]", text))
    if len(text) == 0:
        return False
    return punct/len(text) > 0.3

def repeated_words(text):
    words = text.lower().split()
    for i in range(len(words)-1):
        if words[i] == words[i+1]:
            return True
    return False

def repeated_chars(text):
    return bool(re.search(r"(.)\1{4,}", text))


def non_letter_noise(text):
    letters = sum(c.isalpha() for c in text)
    if len(text) == 0:
        return False
    return letters/len(text) < 0.6



In [ ]:
stats = Counter()
total = 0

INPUT_JSONL = "En\English_5minSeg.jsonl"
TEXT_FIELD = "m_segment_text"

with open(INPUT_JSONL, "r", encoding="utf8") as f:

    for line in f:

        data = json.loads(line)
        text = str(data.get(TEXT_FIELD,"")).strip()

        total += 1

        if repeated_words(text):
            stats["repeated_words"] += 1


        if punctuation_noise(text):
            stats["punctuation_noise"] += 1

        if repeated_chars(text):
            stats["repeated_characters"] += 1

        if non_letter_noise(text):
            stats["non_letter_noise"] += 1

print("\n===== NOISE STATISTICS =====")
print("Total records:", total)

for k,v in stats.items():
    percent = (v/total)*100
    print(f"{k}: {v} ({percent:.2f}%)")

In [ ]:
#with indic nlp inltk


ERROR: Could not find a version that satisfies the requirement intlk (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for intlk


In [40]:
INPUT_JSONL=r'C:\Users\Nita Jadav\Documents\VS_code\proposed3\H\Hindi_5minSeg.jsonl'
te_data=load_jsonl(INPUT_JSONL)

In [41]:
te_data.head()

,cmerge_id,source,video_id,m_segment_id,m_name,m_segment_text
0,1,Hmerge_transcript.csv,1,1,M_1_001,तो भा इस वीडियो पे आपने क्लिक किया। इसका मतलब ...
1,2,Hmerge_transcript.csv,1,2,M_1_002,को लेना चाहिए आप टाइप सकर् टइप कर सक टाइं कर स...
2,3,Hmerge_transcript.csv,1,3,M_1_003,तो अभी आपने यह तो जान लिया कंप्यूटर पार्ट कौन-...
3,4,Hmerge_transcript.csv,1,4,M_1_004,आपको य क करना क्य आपका ग्राफिक आउटुट है। राटटम...
4,5,Hmerge_transcript.csv,1,5,M_1_005,ाडससौन सा बेहर हैगर आपाहकेडिय चप्केो यैटी मनेो...


In [42]:
import re
def clean_text(df, column):
  df[column] = df[column].apply(lambda x: re.sub(r'-\s*\n\s*', '', x))
  df[column] = df[column].apply(lambda x: re.sub(r'\n\s*\n', '\n', x))
  df[column] = df[column].apply(lambda x: re.sub(r'-{2,}', ' ', x))  
  df[column] = df[column].apply(lambda x: re.sub(r'[ \t]+', ' ', x))
  df[column] = df[column].apply(lambda x: re.sub(r'_+', ' ', x))
  df[column] = df[column].apply(lambda x: re.sub(r'\.{2,}', ' ', x))
  df[column] = df[column].apply(lambda x: re.sub(r'\n{2,}', '\n\n', x))
  df[column]=  df[column].apply(lambda x: re.sub(r'([^\n])\n([^\n])', r'\1 \2',x))
  df[column]=df[column].apply(lambda x: re.sub(r'\n\s*\n', '\n', x))
  df[column] = df[column].apply(lambda x: re.sub(r'(.)\1{3,}', r'\1', x))
  df[column] =df[column].apply(lambda x: re.sub(r'([|!?.,;:])\1+', r'\1', x))
  #df[column]=df[column].apply(lambda x: re.sub(r'\b(\w+)(\s+\1\b)+', r'\1', text, flags=re.IGNORECASE)
  #df[column] =df[column].apply(lambda x:x.encode("ascii", "ignore").decode())
  df[column] = df[column].apply(lambda x: re.sub(r"\s+", " ", x))
  return df

In [43]:
te_data=clean_text(te_data, 'm_segment_text')

In [45]:
#sir reference standard format for sentence tokenization in multilanguage
# lang = 0 for languages ['hi', 'or', 'mn', 'as', 'bn', 'pa'], purna biram as sentence end marker
# lang = 1 for Urdu and Kashmiri '۔' sentence end marker
# lang = 2 for languages ['en', 'gu', 'mr', 'ml', 'kn', 'te', 'ta'] '.' sentence end marker
# the mapping of languages and ISO code
# Hindi: hi, Odia: or, Manipuri: mn, Assamese: as, Bengali: bn, Punjabi: pa
# Urdu: ur, Kashmiri: ks
# English: en, Gujarati: gu, Marathi: mr, Malayalam: ml, Kannada: kn, Telugu: te, Tamil: ta
import re
import argparse
import os
from string import punctuation


# the below code defines different kinds of regular expressions
token_specification = [
    ('datemonth',
     r'^(0?[1-9]|1[012])[-\/\.](0?[1-9]|[12][0-9]|3[01])[-\/\.](1|2)\d\d\d$'),
    ('monthdate',
     r'^(0?[1-9]|[12][0-9]|3[01])[-\/\.](0?[1-9]|1[012])[-\/\.](1|2)\d\d\d$'),
    ('yearmonth',
     r'^((1|2)\d\d\d)[-\/\.](0?[1-9]|1[012])[-\/\.](0?[1-9]|[12][0-9]|3[01])'),
    ('EMAIL1', r'([\w\.])+@(\w)+\.(com|org|co\.in)$'),
    ('url1', r'(www\.)([-a-z0-9]+\.)*([-a-z0-9]+.*)(\/[-a-z0-9]+)*/i'),
    ('url', r'/((?:https?\:\/\/|www\.)(?:[-a-z0-9]+\.)*[-a-z0-9]+.*)/i'),
    ('BRACKET', r'[\(\)\[\]\{\}]'),       # Brackets
    ('urdu_year', r'^(ء)(\d{4,4})'),
    ('bullets', r'(\d+\.)$'), # Bullets
    ('NUMBER', r'^(\d+)([,\.٫٬]\d+)*(\S)*'),  # Integer or decimal number
    ('ASSIGN', r'[~:]'),          # Assignment operator
    ('END', r'[;!_]'),           # Statement terminator
    ('EQUAL', r'='),   # Equals
    ('OP', r'[+*\/\-]'),    # Arithmetic operators
    ('QUOTES', r'[\"\'‘’“”]'),          # quotes
    ('Fullstop', r'(\.+)$'),
    ('ellips', r'\.(\.)+'),
    ('HYPHEN', r'[-+\|+]'),
    ('Slashes', r'[\\\/]'),
    ('COMMA12', r'[,%]'),
    ('hin_stop', r'।'),
    ('urdu_stop', r'۔'),
    ('urdu_comma', r'،'),
    ('urdu_semicolon', r'؛'),
    ('urdu_question_mark', r'؟'),
    ('urdu_percent', r'٪'),
    ('quotes_question', r'[”\?]'),
    ('hashtag', r'#'),
    ('join', r'–')
]
# the below code converts the above expression into a python regex
tok_regex = '|'.join('(?P<%s>%s)' % pair for pair in token_specification)
get_token = re.compile(tok_regex, re.U)
punctuations = punctuation + '\"\'‘’“”'


def tokenize(list_s):
    """Tokenize a list of tokens."""
    tkns = []
    for wrds in list_s:
        wrds_len = len(wrds)
        initial_pos = 0
        end_pos = 0
        while initial_pos <= (wrds_len - 1):
            mo = get_token.match(wrds, initial_pos)
            if mo is not None and len(mo.group(0)) == wrds_len:
                if mo.lastgroup == 'urdu_year':
                    tkns.append(wrds[: -4])
                    tkns.append(wrds[-4:])
                else:
                    tkns.append(wrds)
                initial_pos = wrds_len
            else:
                match_out = get_token.search(wrds, initial_pos)
                if match_out is not None:
                    end_pos = match_out.end()
                    if match_out.lastgroup in ["NUMBER", "bullets"]:
                        aa = wrds[initial_pos:(end_pos)]
                    else:
                        aa = wrds[initial_pos:(end_pos - 1)]
                    if aa != '':
                        tkns.append(aa)
                    if match_out.lastgroup not in ["NUMBER", "bullets"]:
                        tkns.append(match_out.group(0))
                    initial_pos = end_pos
                else:
                    tkns.append(wrds[initial_pos:])
                    initial_pos = wrds_len
    return tkns


def read_lines_from_file(file_path):
    """Read lines from a file."""
    with open(file_path, 'r', encoding='utf-8') as file_read:
        return [line.strip() for line in file_read.readlines() if line.strip()]


def proper_bullet_creation(text, pattern):
    """Create proper bullet points after removing spaces between them."""
    text = re.sub('\s{2,}', ' ', text)
    updated_text = ''
    bullet_patterns = re.finditer(pattern, text)
    bullet_patterns = list(bullet_patterns)
    if not bullet_patterns:
        updated_text = text
    else:
        prev_end = -100000
        for bullet_pattern in bullet_patterns:
            start, end = bullet_pattern.start(), bullet_pattern.end()
            if start == prev_end:
                updated_text = updated_text.strip()
                updated_text += bullet_pattern.group(1)
            else:
                updated_text += text[prev_end: start]
                updated_text += bullet_pattern.group(1)
            prev_end = end
        if end != len(text):
            updated_text += text[end:]
    return updated_text


def read_tokenize(lines, lang_type=0):
    """Read a file and tokenize its content by specifying the input file path and language type."""
    # file_read = open(input_file, 'r', encoding='utf-8')
    #lines = read_lines_from_file(input_file)
    pattern = '(\d+\.\s?)'
    lines = [proper_bullet_creation(line, pattern) for line in lines]
    text = '\n'.join(lines)
    if lang_type == 0:
        sentences = re.findall('.*?।|.*?\n', text + '\n', re.UNICODE)
        end_markers = ['?', '।', '!', '|']
    elif lang_type == 1:
        sentences = re.findall('.*?\n', text + '\n', re.UNICODE)
        end_markers = ['؟', '!', '|', '۔']
    else:
        sentences = re.findall('.*?\n', text + '\n', re.UNICODE)
        end_markers = ['?', '.', '!', '|']
    proper_sentences = list()
    for index, sentence in enumerate(sentences):
        sentence = sentence.strip()
        if sentence != '':
            list_tokens = tokenize(sentence.split())
            end_sentence_markers = [index + 1 for index, token in enumerate(list_tokens) if token in end_markers]
            if len(end_sentence_markers) > 0:
                if end_sentence_markers[-1] != len(list_tokens):
                    end_sentence_markers += [len(list_tokens)]
                end_sentence_markers_with_sentence_end_positions = [0] + end_sentence_markers
                sentence_boundaries = list(zip(end_sentence_markers_with_sentence_end_positions, end_sentence_markers_with_sentence_end_positions[1:]))
                for start, end in sentence_boundaries:
                    individual_sentence = list_tokens[start: end]
                    proper_sentences.append(' '.join(individual_sentence))
            else:
                proper_sentences.append(' '.join(list_tokens))
            if index < len(sentences) - 1:
                next_sentence = sentences[index + 1]
                next_tokens = tokenize(next_sentence.split())
                punct_flag = True
                for token in next_tokens:
                    punct_flag &= token in punctuations
                if punct_flag:
                    if proper_sentences:
                        proper_sentences[-1] += ' ' + ' '.join(next_tokens)
                        sentences[index + 1] = ''
    return  proper_sentences


<>:99: SyntaxWarning: invalid escape sequence '\s'
<>:125: SyntaxWarning: invalid escape sequence '\d'
<>:99: SyntaxWarning: invalid escape sequence '\s'
<>:125: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Nita Jadav\AppData\Local\Temp\ipykernel_7492\3942283056.py:99: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s{2,}', ' ', text)
C:\Users\Nita Jadav\AppData\Local\Temp\ipykernel_7492\3942283056.py:125: SyntaxWarning: invalid escape sequence '\d'
  pattern = '(\d+\.\s?)'


In [46]:
def sentence_stats(text):
    if not isinstance(text, str) or not text.strip():
        return 0,0, 0, 0, 0, 0
    
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]

    # Extract sentences and words
    sentences = read_tokenize(lines)
    if not sentences:
        return 0, 0, 0, 0, 0, 0
    
    lengths = []
    total_words = 0

    for sent in sentences:
        sent_tokens = tokenize(sent.split())
        l = len(sent_tokens)
        lengths.append(l)
        total_words += l

    sentence_count = len(sentences)

    return (
        sentences,                # list of sentences
        total_words,              # total tokens
        sentence_count,           # number of sentences
        min(lengths),             # min sentence length
        max(lengths),             # max sentence length
        total_words / sentence_count  # avg length
    )


    return (
        sentences,
        word_count,
        sentence_count,
        min(lengths),
        max(lengths),
        sum(lengths) / sentence_count
    )

In [47]:
te_data[["sentences","word_count", "sentence_count", "min_sent_length",
      "max_sent_length", "avg_sent_length"]] = te_data["m_segment_text"].apply(
    lambda x: pd.Series(sentence_stats(x))
)

In [16]:
#en
print("min word:", te_data['word_count'].min())
print("avg word:", te_data['word_count'].mean())
print("max word:", te_data['word_count'].max())
print("min sentecne:", te_data['sentence_count'].min())
print("avg sentecne:", te_data['sentence_count'].mean())
print("max sentecne:", te_data['sentence_count'].max())
short_count = (te_data["min_sent_length"] < 4).sum()
print("Number of short sentences:", short_count)

min word: 15
avg word: 679.9914928649836
max word: 4816
min sentecne: 1
avg sentecne: 39.026070252469815
max sentecne: 127
Number of short sentences: 1817


In [48]:
te_data.head()

,cmerge_id,source,video_id,m_segment_id,m_name,m_segment_text,sentences,word_count,sentence_count,min_sent_length,max_sent_length,avg_sent_length
0,1,Hmerge_transcript.csv,1,1,M_1_001,तो भा इस वीडियो पे आपने क्लिक किया। इसका मतलब ...,"[तो भा इस वीडियो पे आपने क्लिक किया ।, इसका मत...",899,49,3,48,18.346939
1,2,Hmerge_transcript.csv,1,2,M_1_002,को लेना चाहिए आप टाइप सकर् टइप कर सक टाइं कर स...,[को लेना चाहिए आप टाइप सकर् टइप कर सक टाइं कर ...,959,50,4,54,19.180000
2,3,Hmerge_transcript.csv,1,3,M_1_003,तो अभी आपने यह तो जान लिया कंप्यूटर पार्ट कौन-...,[तो अभी आपने यह तो जान लिया कंप्यूटर पार्ट कौन...,625,33,3,49,18.939394
3,4,Hmerge_transcript.csv,1,4,M_1_004,आपको य क करना क्य आपका ग्राफिक आउटुट है। राटटम...,"[आपको य क करना क्य आपका ग्राफिक आउटुट है ।, रा...",698,49,4,40,14.244898
4,5,Hmerge_transcript.csv,1,5,M_1_005,ाडससौन सा बेहर हैगर आपाहकेडिय चप्केो यैटी मनेो...,[ाडससौन सा बेहर हैगर आपाहकेडिय चप्केो यैटी मने...,731,37,4,54,19.756757


In [23]:
#gu
print("min word:", te_data['word_count'].min())
print("avg word:", te_data['word_count'].mean())
print("max word:", te_data['word_count'].max())
print("min sentecne:", te_data['sentence_count'].min())
print("avg sentecne:", te_data['sentence_count'].mean())
print("max sentecne:", te_data['sentence_count'].max())
short_count = (te_data["min_sent_length"] < 4).sum()
print("Number of short sentences:", short_count)

min word: 2
avg word: 470.4588364434687
max word: 1543
min sentecne: 1
avg sentecne: 42.80186608122942
max sentecne: 350
Number of short sentences: 2365


In [37]:
#te
print("min word:", te_data['word_count'].min())
print("avg word:", te_data['word_count'].mean())
print("max word:", te_data['word_count'].max())
print("min sentecne:", te_data['sentence_count'].min())
print("avg sentecne:", te_data['sentence_count'].mean())
print("max sentecne:", te_data['sentence_count'].max())
short_count = (te_data["min_sent_length"] < 4).sum()
print("Number of short sentences:", short_count)

min word: 2
avg word: 369.1661971830986
max word: 1275
min sentecne: 1
avg sentecne: 33.67492957746479
max sentecne: 227
Number of short sentences: 2132


In [38]:
short_word_records = te_data[te_data["word_count"] == 2]

In [39]:
short_word_records

,cmerge_id,source,video_id,m_segment_id,m_name,m_segment_text,sentences,word_count,sentence_count,min_sent_length,max_sent_length,avg_sent_length
3486,3487,Ta2Temerge_transcript.csv,44,178,B_44_003,పెరియా?,[పెరియా ?],2,1,2,2,2.0
3520,3521,Ta2Temerge_transcript.csv,57,212,B_57_003,అదే విధంగా,[అదే విధంగా],2,1,2,2,2.0


In [49]:
#hn
print("min word:", te_data['word_count'].min())
print("avg word:", te_data['word_count'].mean())
print("max word:", te_data['word_count'].max())
print("min sentecne:", te_data['sentence_count'].min())
print("avg sentecne:", te_data['sentence_count'].mean())
print("max sentecne:", te_data['sentence_count'].max())
short_count = (te_data["min_sent_length"] < 4).sum()
print("Number of short sentences:", short_count)

min word: 3
avg word: 498.43715697036225
max word: 1702
min sentecne: 1
avg sentecne: 27.760428100987927
max sentecne: 144
Number of short sentences: 1159


In [50]:
short_word_records = te_data[te_data["word_count"] == 3]

In [51]:
short_word_records

,cmerge_id,source,video_id,m_segment_id,m_name,m_segment_text,sentences,word_count,sentence_count,min_sent_length,max_sent_length,avg_sent_length
3389,3390,Ta2Hmerge_translated.csv,57,212,B_57_003,यह वही है,[यह वही है],3,1,3,3,3.0
